In [ ]:
from pathlib import Path

# Estructura base del repo para ejecucion reproducible
Path('datasets').mkdir(exist_ok=True)
Path('datasets/dataset_downloads').mkdir(parents=True, exist_ok=True)
Path('analytics').mkdir(exist_ok=True)
Path('docs').mkdir(exist_ok=True)
print('Estructura verificada: datasets/, datasets/dataset_downloads/, analytics/, docs/')

# RecopilaciÃƒÂ³n de Datasets de CaÃƒÂ­das y Sensores Inerciales (Wearables)

Este notebook mejora la descarga y acceso a conjuntos de datos de detecciÃƒÂ³n de caÃƒÂ­das. Se ha implementado:
1. **Manejo robusto de credenciales de Kaggle** (necesarias para varios de los datasets mÃƒÂ¡s importantes).
2. **InclusiÃƒÂ³n de nuevos datasets**, como *UP-Fall Detection* y *Real-Time Patient Fall Detection*.
3. **Manejo de excepciones** para enlaces directos que puedan estar caÃƒÂ­dos.


In [1]:
!pip install -q pandas numpy requests kagglehub wfdb
import pandas as pd
import requests
import os
import io
import kagglehub
print("LibrerÃƒÂ­as instaladas y listas.")


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\juane\OneDrive\Documentos\UniversidadEAN\ProyectoGrado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LibrerÃƒÂ­as instaladas y listas.


## 1. ConfiguraciÃƒÂ³n de Credenciales de Kaggle
Muchos datasets de caÃƒÂ­das en Kaggle requieren autenticaciÃƒÂ³n para descargarse por API.

Esta celda ahora funciona tanto en **Google Colab** (leyendo Secrets) como en ejecuciÃƒÂ³n local en VS Code/Jupyter.
- Si ya tienes `KAGGLE_USERNAME` y `KAGGLE_KEY` en variables de entorno, las reutiliza.
- Si estÃƒÂ¡s en Colab, intenta leerlas desde `userdata`.
- Si no existen, te pedirÃƒÂ¡ ingresarlas manualmente para la sesiÃƒÂ³n actual.

In [11]:
import getpass
import os
from pathlib import Path

def _has_any_kaggle_auth():
    has_legacy = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_token = bool(os.environ.get("KAGGLE_API_TOKEN"))
    return has_legacy or has_token

# 1) Try loading from local .env (KAGGLE_API -> KAGGLE_API_TOKEN)
env_path = Path('.env')
if env_path.exists() and not os.environ.get('KAGGLE_API_TOKEN'):
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k == 'KAGGLE_API' and v:
            os.environ['KAGGLE_API_TOKEN'] = v

if _has_any_kaggle_auth():
    print("Credenciales de Kaggle detectadas (legacy o token API).")
else:
    # 2) Try Colab secrets
    try:
        from google.colab import userdata
        token = userdata.get("KAGGLE_API_TOKEN")
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
        else:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Credenciales cargadas desde Colab Secrets.")
    except Exception:
        print("No se detectaron credenciales automÃƒÂ¡ticas. Ingresa credenciales manualmente.")
        use_token = input("Ã‚Â¿Tienes KAGGLE_API_TOKEN? (s/n): ").strip().lower()
        if use_token == 's':
            os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN: ").strip()
        else:
            os.environ["KAGGLE_USERNAME"] = input("KAGGLE_USERNAME: ").strip()
            os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ").strip()
        print("Credenciales configuradas temporalmente para esta sesiÃƒÂ³n.")

Credenciales de Kaggle detectadas (legacy o token API).


## 2. Descarga de Datasets vÃƒÂ­a KaggleHub
Se prueban rutas actualizadas y alternativas por dataset. Si una ruta falla, se intenta la siguiente.

Nota: algunos mirrors en Kaggle no son oficiales del paper original. Conviene revisar licencia/procedencia antes de entrenar modelos finales.

In [5]:
datasets_kaggle = {
    "FallAllD (mirror Kaggle)": [
        "shusrith/fallalld",
        "kofischolar/fallalld-dataset"
    ],
    "KFall (mirror Kaggle)": [
        "usmanabbasi2002/kfall-dataset",
        "kushajmallick/kfall-fulldata"
    ],
    "Real-Time Patient Fall": [
        "zara2099/real-time-patient-fall-detection-data"
    ],
    "SisFall (mirror Kaggle)": [
        "thevman/sisfall-dataset",
        "adityavvvn/sisfall"
    ]
}

for dataset_name, candidates in datasets_kaggle.items():
    print(f"\n=== {dataset_name} ===")
    downloaded = False

    for slug in candidates:
        print(f"Intentando: {slug}")
        try:
            dataset_path = kagglehub.dataset_download(slug)
            print(f"Ã¢Å“â€œ Descarga exitosa: {dataset_path}")

            csv_files = []
            for root, _, files in os.walk(dataset_path):
                for file in files:
                    if file.lower().endswith(".csv"):
                        csv_files.append(os.path.join(root, file))

            if csv_files:
                sample_file = csv_files[0]
                print(f"Ã¢Å“â€œ CSV detectado: {os.path.basename(sample_file)}")
                try:
                    df_temp = pd.read_csv(sample_file)
                except UnicodeDecodeError:
                    df_temp = pd.read_csv(sample_file, encoding="latin1")
                print(f"  Shape: {df_temp.shape}")
                display(df_temp.head(3))
            else:
                print("Ã¢â€žÂ¹ No se detectaron CSV directos (podrÃƒÂ­a venir en .txt/.mat/.zip).")

            downloaded = True
            break
        except Exception as e:
            print(f"Ã¢Å“â€” FallÃƒÂ³ {slug}: {e}")

    if not downloaded:
        print("Ã¢Å¡Â  No fue posible descargar este dataset con los slugs configurados.")


=== FallAllD (mirror Kaggle) ===
Intentando: shusrith/fallalld


100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 853M/853M [00:24<00:00, 37.0MB/s] 

Extracting files...


Ã¢Å“â€” FallÃƒÂ³ shusrith/fallalld: [Errno 2] No such file or directory: 'C:\\Users\\juane\\.cache\\kagglehub\\datasets\\shusrith\\fallalld\\versions\\5\\Activity_DataSet(1)\\ARS DLR Data Set\\QuantizationFiles\\4th_Quantization_AOOG_Individual_Training_4XValidation\\Falling_Part1of4_Maria_SampUpd_25_LastQuantization_Best_Features_Groundtruth.txt'
Intentando: kofischolar/fallalld-dataset


100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 324M/324M [00:09<00:00, 35.8MB/s] 

...


=== SisFall (mirror Kaggle) ===
Intentando: thevman/sisfall-dataset


100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 553M/553M [00:13<00:00, 42.3MB/s] 

Extracting files...


Ã¢Å“â€œ Descarga exitosa: C:\Users\juane\.cache\kagglehub\datasets\thevman\sisfall-dataset\versions\1
Ã¢Å“â€œ CSV detectado: D01_SA01_R01.csv
  Shape: (5401, 10)


,ADXL345_Acc_X,ADXL345_Acc_Y,ADXL345_Acc_Z,ITG3200_Gyro_X,ITG3200_Gyro_Y,ITG3200_Gyro_Z,MMA8451Q_Acc_X,MMA8451Q_Acc_Y,MMA8451Q_Acc_Z,FallCheck
0,0.058554,-0.679685,-0.351576,-3.233633,-34.665666,-18.676268,0.046844,-0.659182,-0.248050,0
1,-0.043588,-0.730261,-0.302868,-5.400030,-38.734197,-13.896265,-0.032360,-0.702897,-0.180819,0
2,-0.129813,-0.862197,-0.233726,-9.369714,-42.680881,-9.822588,-0.107507,-0.829785,-0.105658,0


## 3. VerificaciÃƒÂ³n de Enlaces Directos (GitHub/HTTP)
Varios enlaces raw histÃƒÂ³ricos pueden romperse con el tiempo. Primero validamos estado HTTP y solo leemos CSV si el endpoint responde correctamente.

In [4]:
import requests

urls_directas = {
    "UR Fall Detection (raw CSV histÃƒÂ³rico)": "https://raw.githubusercontent.com/czhao30/UR-Fall-Detection-Dataset/master/urfall-01/urfall-01-acc.csv",
    "UniMiB SHAR (raw CSV histÃƒÂ³rico)": "https://raw.githubusercontent.com/jindongwang/transferlearning/master/code/deep/Mobile/UniMiB_SHAR/data/full_data.csv"
}

for name, url in urls_directas.items():
    print(f"\n=== Verificando {name} ===")
    try:
        resp = requests.get(url, timeout=20)
        status = resp.status_code
        print(f"Estado HTTP: {status}")

        if status == 200:
            df_direct = pd.read_csv(io.StringIO(resp.text), header=None)
            print(f"Ã¢Å“â€œ Lectura exitosa. Shape: {df_direct.shape}")
            display(df_direct.head(3))
        elif status == 404:
            print("Ã¢Å¡Â  Enlace roto (404). Busca una fuente alternativa o mirror actualizado.")
        else:
            print("Ã¢Å¡Â  Endpoint no disponible para descarga directa en este momento.")
    except Exception as e:
        print(f"Ã¢Å“â€” Error verificando {name}: {e}")

print("\nSugerencia: para estos dos datasets, prioriza Kaggle mirrors validados o la fuente oficial del paper.")


=== Verificando UR Fall Detection (raw CSV histÃƒÂ³rico) ===
Estado HTTP: 404
Ã¢Å¡Â  Enlace roto (404). Busca una fuente alternativa o mirror actualizado.

=== Verificando UniMiB SHAR (raw CSV histÃƒÂ³rico) ===
Estado HTTP: 404
Ã¢Å¡Â  Enlace roto (404). Busca una fuente alternativa o mirror actualizado.

Sugerencia: para estos dos datasets, prioriza Kaggle mirrors validados o la fuente oficial del paper.


## 4. Nuevo Dataset Multimodal: UP-Fall Detection Dataset
El **UP-Fall Detection Dataset** es una de las opciones mÃƒÂ¡s exhaustivas y modernas.
- **Sensores incluidos:** AcelerÃƒÂ³metro, giroscopio, luz ambiental (mediante 5 wearables), cÃƒÂ¡maras, sensores infrarrojos e incluso cascos EEG.
- **Volumen de datos:** Debido a su enfoque multimodal, el dataset consolidado pesa mÃƒÂ¡s de **812 GB** y los features extraÃƒÂ­dos unos **171 GB**.
- **Sujetos:** 17 voluntarios sanos realizando 11 actividades distintas (6 actividades diarias simples y 5 tipos de caÃƒÂ­das).

*Por motivos de tamaÃƒÂ±o, este dataset no se descarga directamente a la memoria RAM de Colab mediante un CSV. Para usarlo, debes descargar las particiones o features tabulares desde su repositorio oficial o buscar subconjuntos especÃƒÂ­ficos (como los features en CSV) dependiendo de si quieres procesar imÃƒÂ¡genes, series de tiempo inerciales o seÃƒÂ±ales cerebrales.*

**Acceso Oficial:** [UP-Fall Detection Dataset (Sitio Oficial)](https://sites.google.com/up.edu.mx/har-up/)

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

upfall_dir = Path("datasets/UP-Fall")
csv_paths = sorted(upfall_dir.glob("*.csv"))

summary_rows = []
for p in csv_paths:
    # Read full file to get robust stats; low_memory avoids dtype fragmentation warnings.
    df = pd.read_csv(p, low_memory=False)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    object_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    null_cells = int(df.isna().sum().sum())
    duplicated_rows = int(df.duplicated().sum())

    summary_rows.append({
        "file": p.name,
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "numeric_cols": len(numeric_cols),
        "object_cols": len(object_cols),
        "null_cells": null_cells,
        "duplicate_rows": duplicated_rows
    })

summary_df = pd.DataFrame(summary_rows).sort_values(by=["rows", "cols"], ascending=False)
print("=== Resumen general de archivos UP-Fall ===")
display(summary_df)

total_rows = int(summary_df["rows"].sum())
print(f"Total de filas combinadas (sumando todos los CSV): {total_rows:,}")

# Quick label audit for candidate columns if present.
label_candidates = ["Activity", "activity", "Label", "label", "Fall", "fall", "fall_event", "posture"]
print("\n=== AuditorÃƒÂ­a de columnas potenciales de etiqueta ===")
for p in csv_paths:
    df = pd.read_csv(p, low_memory=False)
    present = [c for c in label_candidates if c in df.columns]
    if present:
        print(f"\n{p.name}")
        for c in present:
            vc = df[c].value_counts(dropna=False).head(10)
            print(f"- {c} (top 10):")
            display(vc)
    else:
        print(f"{p.name}: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.")

=== Resumen general de archivos UP-Fall ===


,file,rows,cols,numeric_cols,object_cols,null_cells,duplicate_rows
1,CompleteDataSet.csv,294679,47,11,36,253230,0
2,CompleteDataSetFeatures1&0.5.csv,32295,761,0,761,575510,0
5,Features_1&0.5_Vision.csv,32237,805,0,805,0,0
0,CameraResizedOF.csv,17378,805,804,1,0,0
3,CompleteDataSetFeatures2&1.csv,15593,761,0,761,269450,0
6,Features_2&1_Vision.csv,15565,805,0,805,0,0
4,CompleteDataSetFeatures3&1.5.csv,10125,761,0,761,173212,0
7,Features_3&1.5_Vision.csv,10107,805,0,805,0,0


Total de filas combinadas (sumando todos los CSV): 427,979

=== AuditorÃƒÂ­a de columnas potenciales de etiqueta ===

CameraResizedOF.csv
- Activity (top 10):


Activity
6     3251
8     3158
11    3136
7     3011
10    1596
3      551
2      548
4      546
5      536
9      534
Name: count, dtype: int64


CompleteDataSet.csv
- Activity (top 10):


Activity
6.0     54568
7.0     54032
8.0     53689
11.0    50872
10.0    26694
4.0      9248
3.0      9237
1.0      9204
2.0      9188
5.0      9074
Name: count, dtype: int64

CompleteDataSetFeatures1&0.5.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.
CompleteDataSetFeatures2&1.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.
CompleteDataSetFeatures3&1.5.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.
Features_1&0.5_Vision.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.
Features_2&1_Vision.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.
Features_3&1.5_Vision.csv: no se detectaron etiquetas en el conjunto estÃƒÂ¡ndar de nombres.


In [7]:
from pathlib import Path
import csv

upfall_dir = Path("datasets/UP-Fall")
csv_paths = sorted(upfall_dir.glob("*.csv"))

print("=== InspecciÃƒÂ³n de estructura (delimiter, encabezados y muestra) ===")
for p in csv_paths:
    with open(p, "r", encoding="utf-8", errors="replace") as f:
        sample = f.read(4096)
        f.seek(0)
        first_line = f.readline().strip()
    
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t", "|"] )
        delim = repr(dialect.delimiter)
    except Exception:
        delim = "'unknown'"

    df0 = pd.read_csv(p, nrows=3, low_memory=False)
    print(f"\nArchivo: {p.name}")
    print(f"- Delimitador detectado: {delim}")
    print(f"- Columnas: {len(df0.columns)}")
    print(f"- Primeras 8 columnas: {df0.columns[:8].tolist()}")
    print(f"- Primera lÃƒÂ­nea (truncada 180): {first_line[:180]}")

    # Try to coerce object columns to numeric and estimate conversion ratio.
    obj_cols = df0.select_dtypes(include=['object']).columns.tolist()
    if obj_cols:
        conv_ok = 0
        for c in obj_cols:
            s = pd.to_numeric(df0[c].astype(str).str.strip(), errors="coerce")
            if s.notna().mean() >= 0.66:
                conv_ok += 1
        print(f"- Columnas object en muestra: {len(obj_cols)}; potencialmente numÃƒÂ©ricas: {conv_ok}")
    else:
        print("- Sin columnas object en muestra.")

=== InspecciÃƒÂ³n de estructura (delimiter, encabezados y muestra) ===

Archivo: CameraResizedOF.csv
- Delimitador detectado: ','
- Columnas: 805
- Primeras 8 columnas: ['Timestamp', 'C1(0;0)', 'C1(0;1)', 'C1(0;2)', 'C1(0;3)', 'C1(0;4)', 'C1(0;5)', 'C1(0;6)']
- Primera lÃƒÂ­nea (truncada 180): Timestamp,C1(0;0),C1(0;1),C1(0;2),C1(0;3),C1(0;4),C1(0;5),C1(0;6),C1(0;7),C1(0;8),C1(0;9),C1(0;10),C1(0;11),C1(0;12),C1(0;13),C1(0;14),C1(0;15),C1(0;16),C1(0;17),C1(0;18),C1(0;19),
- Columnas object en muestra: 1; potencialmente numÃƒÂ©ricas: 0

Archivo: CompleteDataSet.csv
- Delimitador detectado: ','
- Columnas: 47
- Primeras 8 columnas: ['TimeStamps', 'AnkleAccelerometer', 'Unnamed: 2', 'Unnamed: 3', 'AnkleAngularVelocity', 'Unnamed: 5', 'Unnamed: 6', 'AnkleLuminosity']
- Primera lÃƒÂ­nea (truncada 180): TimeStamps,AnkleAccelerometer,,,AnkleAngularVelocity,,,AnkleLuminosity,RightPocketAccelerometer,,,RightPocketAngularVelocity,,,RightPocketLuminosity,BeltAccelerometer,,,BeltAngular
- Columnas 

C:\Users\juane\AppData\Local\Temp\ipykernel_4772\456265386.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df0.select_dtypes(include=['object']).columns.tolist()
C:\Users\juane\AppData\Local\Temp\ipykernel_4772\456265386.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

- Columnas object en muestra: 761; potencialmente numÃƒÂ©ricas: 756

Archivo: CompleteDataSetFeatures2&1.csv
- Delimitador detectado: 'unknown'
- Columnas: 761
- Primeras 8 columnas: ['Unnamed: 0', 'AnkleAccelerometer: x-axis (g)', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']
- Primera lÃƒÂ­nea (truncada 180): ,AnkleAccelerometer: x-axis (g),,,,,,,,,,,,,,,,,,AnkleAccelerometer: y-axis (g),,,,,,,,,,,,,,,,,,AnkleAccelerometer: z-axis (g),,,,,,,,,,,,,,,,,,AnkleAngularVelocity: x-axis (deg/s


C:\Users\juane\AppData\Local\Temp\ipykernel_4772\456265386.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df0.select_dtypes(include=['object']).columns.tolist()


- Columnas object en muestra: 761; potencialmente numÃƒÂ©ricas: 756

Archivo: CompleteDataSetFeatures3&1.5.csv
- Delimitador detectado: 'unknown'
- Columnas: 761
- Primeras 8 columnas: ['Unnamed: 0', 'AnkleAccelerometer: x-axis (g)', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']
- Primera lÃƒÂ­nea (truncada 180): ,AnkleAccelerometer: x-axis (g),,,,,,,,,,,,,,,,,,AnkleAccelerometer: y-axis (g),,,,,,,,,,,,,,,,,,AnkleAccelerometer: z-axis (g),,,,,,,,,,,,,,,,,,AnkleAngularVelocity: x-axis (deg/s


...

- Columnas object en muestra: 805; potencialmente numÃƒÂ©ricas: 804

Archivo: Features_2&1_Vision.csv
- Delimitador detectado: ','
- Columnas: 805
- Primeras 8 columnas: ['Unnamed: 0', 'C1(0;0)', 'C1(0;1)', 'C1(0;2)', 'C1(0;3)', 'C1(0;4)', 'C1(0;5)', 'C1(0;6)']
- Primera lÃƒÂ­nea (truncada 180): ,C1(0;0),C1(0;1),C1(0;2),C1(0;3),C1(0;4),C1(0;5),C1(0;6),C1(0;7),C1(0;8),C1(0;9),C1(0;10),C1(0;11),C1(0;12),C1(0;13),C1(0;14),C1(0;15),C1(0;16),C1(0;17),C1(0;18),C1(0;19),C1(1;0),C


C:\Users\juane\AppData\Local\Temp\ipykernel_4772\456265386.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df0.select_dtypes(include=['object']).columns.tolist()


- Columnas object en muestra: 805; potencialmente numÃƒÂ©ricas: 804

Archivo: Features_3&1.5_Vision.csv
- Delimitador detectado: ','
- Columnas: 805
- Primeras 8 columnas: ['Unnamed: 0', 'C1(0;0)', 'C1(0;1)', 'C1(0;2)', 'C1(0;3)', 'C1(0;4)', 'C1(0;5)', 'C1(0;6)']
- Primera lÃƒÂ­nea (truncada 180): ,C1(0;0),C1(0;1),C1(0;2),C1(0;3),C1(0;4),C1(0;5),C1(0;6),C1(0;7),C1(0;8),C1(0;9),C1(0;10),C1(0;11),C1(0;12),C1(0;13),C1(0;14),C1(0;15),C1(0;16),C1(0;17),C1(0;18),C1(0;19),C1(1;0),C


C:\Users\juane\AppData\Local\Temp\ipykernel_4772\456265386.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df0.select_dtypes(include=['object']).columns.tolist()


- Columnas object en muestra: 805; potencialmente numÃƒÂ©ricas: 804


In [8]:
from pathlib import Path
import os
import kagglehub

# Carga token desde .env sin imprimir secretos
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key == 'KAGGLE_API' and value:
            os.environ['KAGGLE_API_TOKEN'] = value

print('KAGGLE_API_TOKEN presente:', bool(os.environ.get('KAGGLE_API_TOKEN')))

slugs_test = [
    'pabloInercial/fallalld',
    'kfall-dataset/kfall',
    'mohitkr05/sisfall-dataset',
    'zara2099/real-time-patient-fall-detection-data',
    'shusrith/fallalld',
    'usmanabbasi2002/kfall-dataset',
    'thevman/sisfall-dataset'
]

results = []
for slug in slugs_test:
    try:
        path = kagglehub.dataset_download(slug)
        results.append((slug, 'OK', path))
        print(f'OK: {slug}')
    except Exception as e:
        msg = str(e).splitlines()[0][:240]
        results.append((slug, 'ERROR', msg))
        print(f'ERROR: {slug} -> {msg}')

results

KAGGLE_API_TOKEN presente: True
ERROR: pabloInercial/fallalld -> 403 Client Error.
ERROR: kfall-dataset/kfall -> 403 Client Error.
ERROR: mohitkr05/sisfall-dataset -> 403 Client Error.
OK: zara2099/real-time-patient-fall-detection-data
Download already complete (894324299 bytes).
Extracting files...
ERROR: shusrith/fallalld -> [Errno 2] No such file or directory: 'C:\\Users\\juane\\.cache\\kagglehub\\datasets\\shusrith\\fallalld\\versions\\5\\Activity_DataSet(1)\\ARS DLR Data Set\\QuantizationFiles\\4th_Quantization_AOOG_Individual_Training_4XValidation\\Falling_
OK: usmanabbasi2002/kfall-dataset
OK: thevman/sisfall-dataset


[('pabloInercial/fallalld', 'ERROR', '403 Client Error.'),
 ('kfall-dataset/kfall', 'ERROR', '403 Client Error.'),
 ('mohitkr05/sisfall-dataset', 'ERROR', '403 Client Error.'),
 ('zara2099/real-time-patient-fall-detection-data',
  'OK',
  'C:\\Users\\juane\\.cache\\kagglehub\\datasets\\zara2099\\real-time-patient-fall-detection-data\\versions\\1'),
 ('shusrith/fallalld',
  'ERROR',
  "[Errno 2] No such file or directory: 'C:\\\\Users\\\\juane\\\\.cache\\\\kagglehub\\\\datasets\\\\shusrith\\\\fallalld\\\\versions\\\\5\\\\Activity_DataSet(1)\\\\ARS DLR Data Set\\\\QuantizationFiles\\\\4th_Quantization_AOOG_Individual_Training_4XValidation\\\\Falling_"),
 ('usmanabbasi2002/kfall-dataset',
  'OK',
  'C:\\Users\\juane\\.cache\\kagglehub\\datasets\\usmanabbasi2002\\kfall-dataset\\versions\\2'),
 ('thevman/sisfall-dataset',
  'OK',
  'C:\\Users\\juane\\.cache\\kagglehub\\datasets\\thevman\\sisfall-dataset\\versions\\1')]

In [9]:
from pathlib import Path
import os
import kagglehub

# Workaround Windows path length for large nested datasets
short_cache = Path('C:/khcache')
short_cache.mkdir(parents=True, exist_ok=True)
os.environ['KAGGLEHUB_CACHE'] = str(short_cache)
print('KAGGLEHUB_CACHE =', os.environ['KAGGLEHUB_CACHE'])

try:
    path_fallalld = kagglehub.dataset_download('shusrith/fallalld', force_download=True)
    print('FallAllD mirror descargado OK en cache corta:', path_fallalld)
except Exception as e:
    print('FallAllD mirror sigue fallando:', str(e)[:350])

KAGGLEHUB_CACHE = C:\khcache


100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 853M/853M [00:20<00:00, 42.9MB/s] 

Extracting files...


FallAllD mirror descargado OK en cache corta: C:\khcache\datasets\shusrith\fallalld\versions\5


In [10]:
import pandas as pd
from pathlib import Path

upfall_dir = Path('datasets/UP-Fall')
df_complete = pd.read_csv(upfall_dir / 'CompleteDataSet.csv', low_memory=False)
df_camera = pd.read_csv(upfall_dir / 'CameraResizedOF.csv', low_memory=False)

print('=== DistribuciÃƒÂ³n de Activity en CompleteDataSet.csv ===')
if 'Activity' in df_complete.columns:
    vc_complete = df_complete['Activity'].value_counts(dropna=False).sort_index()
    display(vc_complete)
    print('Actividades distintas:', df_complete['Activity'].nunique(dropna=True))
    print('Filas con Activity nula:', int(df_complete['Activity'].isna().sum()))
else:
    print('No existe columna Activity en CompleteDataSet.csv')

print('\n=== DistribuciÃƒÂ³n de Activity en CameraResizedOF.csv ===')
if 'Activity' in df_camera.columns:
    vc_camera = df_camera['Activity'].value_counts(dropna=False).sort_index()
    display(vc_camera)
    print('Actividades distintas:', df_camera['Activity'].nunique(dropna=True))
    print('Filas con Activity nula:', int(df_camera['Activity'].isna().sum()))
else:
    print('No existe columna Activity en CameraResizedOF.csv')

=== DistribuciÃƒÂ³n de Activity en CompleteDataSet.csv ===


Activity
1.0      9204
2.0      9188
3.0      9237
4.0      9248
5.0      9074
6.0     54568
7.0     54032
8.0     53689
9.0      8872
10.0    26694
11.0    50872
NaN         1
Name: count, dtype: int64

Actividades distintas: 11
Filas con Activity nula: 1

=== DistribuciÃƒÂ³n de Activity en CameraResizedOF.csv ===


Activity
1      511
2      548
3      551
4      546
5      536
6     3251
7     3011
8     3158
9      534
10    1596
11    3136
Name: count, dtype: int64

Actividades distintas: 11
Filas con Activity nula: 0


In [12]:
from pathlib import Path
import requests

hf_token = None
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if '=' not in line:
            continue
        k, v = line.split('=', 1)
        if k.strip() == 'HF_TOKEN':
            hf_token = v.strip().strip('"').strip("'")
            break

if hf_token:
    headers = {'Authorization': f'Bearer {hf_token}'}
    r = requests.get('https://huggingface.co/api/whoami-v2', headers=headers, timeout=20)
    print('HF token status code:', r.status_code)
    if r.status_code == 200:
        print('HF token operativo para consultas autenticadas.')
    else:
        print('HF token no validÃƒÂ³ correctamente en whoami-v2.')
else:
    print('HF_TOKEN no encontrado en .env')

HF token status code: 200
HF token operativo para consultas autenticadas.


## 5. Catalogo unificado y validacion automatica

Esta seccion usa un modulo reutilizable para:
- centralizar datasets nucleares y adicionales,
- validar disponibilidad de URLs (oficiales y mirrors),
- generar un resumen de estados para priorizar descargas.

In [ ]:
from dataset_catalog_utils import (
    CORE_DATASETS,
    ADDITIONAL_DATASETS,
    to_dataframe,
    validate_catalog,
    availability_summary,
)

catalog_df = to_dataframe(CORE_DATASETS + ADDITIONAL_DATASETS)
print(f"Total datasets catalogados: {len(catalog_df)}")
display(catalog_df[["name", "source_type", "access_type", "modalities", "sensors", "base_url"]])

validation_df = validate_catalog(CORE_DATASETS + ADDITIONAL_DATASETS, timeout=20)
print("\nResumen de disponibilidad:")
display(availability_summary(validation_df))

print("\nDetalle de validacion:")
display(
    validation_df[[
        "name",
        "availability",
        "base_status",
        "mirror_status",
        "base_url",
        "mirror_url",
        "notes",
    ]]
)

# Guardado opcional para seguimiento historico
validation_df.to_csv("analytics/dataset_validation_report.csv", index=False)
print("Reporte guardado en analytics/dataset_validation_report.csv")

## 6. Validacion de shapes por dataset y consolidacion para reporte

Esta seccion:
- verifica si lo local de UP-Fall corresponde a un subconjunto o al dataset completo,
- calcula filas/columnas por archivo y por dataset cuando sea posible,
- intenta descarga en carpeta local para datasets con acceso por Kaggle,
- exporta resultados para actualizar el reporte markdown.

In [1]:
from pathlib import Path
import csv
import os
import shutil
import pandas as pd
import kagglehub

project_root = Path.cwd()
download_root = project_root / "datasets" / "dataset_downloads"
download_root.mkdir(exist_ok=True)

# Reutilizar credenciales desde .env si existen
env_path = project_root / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        if '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k == 'KAGGLE_API' and v and not os.environ.get('KAGGLE_API_TOKEN'):
            os.environ['KAGGLE_API_TOKEN'] = v
        elif k in {'KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'} and v and not os.environ.get(k):
            os.environ[k] = v

has_kaggle_auth = bool(os.environ.get('KAGGLE_API_TOKEN') or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')))
print('Kaggle auth detectado:', has_kaggle_auth)


def detect_delimiter(file_path: Path) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            sample = f.read(4096)
        dialect = csv.Sniffer().sniff(sample, delimiters=[',', ';', '\t', '|'])
        return dialect.delimiter
    except Exception:
        return ','


def csv_shape_fast(file_path: Path):
    delim = detect_delimiter(file_path)
    rows = 0
    cols = 0
    header = False
    with open(file_path, 'r', encoding='utf-8', errors='replace', newline='') as f:
        reader = csv.reader(f, delimiter=delim)
        for i, row in enumerate(reader):
            if i == 0:
                cols = len(row)
                header = any(not c.replace('.', '', 1).replace('-', '', 1).isdigit() for c in row if c != '')
            rows += 1
    if rows > 0 and header:
        rows -= 1
    return rows, cols


# 1) UP-Fall local: shape por archivo (esto valida claramente que tienes CSV de "complete/features",
# pero no todo el universo de imagenes/camaras del sitio oficial).
upfall_rows = []
upfall_dir = project_root / 'datasets' / 'UP-Fall'
if upfall_dir.exists():
    for p in sorted(upfall_dir.glob('*.csv')):
        try:
            r, c = csv_shape_fast(p)
            upfall_rows.append({'dataset': 'UP-Fall (local)', 'file': p.name, 'rows': r, 'cols': c})
        except Exception as e:
            upfall_rows.append({'dataset': 'UP-Fall (local)', 'file': p.name, 'rows': None, 'cols': None, 'error': str(e)[:180]})

upfall_shapes_df = pd.DataFrame(upfall_rows)
print('\nUP-Fall local files detectados:', len(upfall_shapes_df))
display(upfall_shapes_df)


# 2) Datasets Kaggle para descarga local y resumen de shapes
kaggle_targets = {
    'SisFall (mirror Kaggle)': ['thevman/sisfall-dataset', 'adityavvvn/sisfall'],
    'KFall (mirror Kaggle)': ['usmanabbasi2002/kfall-dataset', 'kushajmallick/kfall-fulldata'],
    'FallAllD (mirror Kaggle)': ['shusrith/fallalld', 'kofischolar/fallalld-dataset'],
    'Real-Time Patient Fall': ['zara2099/real-time-patient-fall-detection-data'],
    'Elderly Fall Detection IoT': ['ziya07/elderly-fall-detection-iot-dataset'],
    'Smartphone Human Fall Dataset': ['saadmansakib/smartphone-human-fall-dataset'],
    'Falls vs Normal Activities': ['enricogrimaldi/falls-vs-normal-activities'],
}

shape_summary_rows = []

if has_kaggle_auth:
    short_cache = Path('C:/khcache')
    short_cache.mkdir(parents=True, exist_ok=True)
    os.environ['KAGGLEHUB_CACHE'] = str(short_cache)

    for dataset_name, slugs in kaggle_targets.items():
        ok = False
        err_msg = ''
        for slug in slugs:
            try:
                path = Path(kagglehub.dataset_download(slug))

                # Copia referencial opcional a carpeta local del proyecto
                target = download_root / slug.replace('/', '__')
                target.mkdir(parents=True, exist_ok=True)

                csv_files = sorted(path.rglob('*.csv'))
                total_rows = 0
                cols_list = []
                sample_file = ''
                sample_shape = 'N/D'

                for i, fcsv in enumerate(csv_files):
                    r, c = csv_shape_fast(fcsv)
                    total_rows += int(r)
                    cols_list.append(int(c))
                    if i == 0:
                        sample_file = str(fcsv.relative_to(path)).replace('\\', '/')
                        sample_shape = f'({r}, {c})'

                # Crear marcador en carpeta local con referencia al cache real
                (target / 'SOURCE_PATH.txt').write_text(str(path), encoding='utf-8')

                shape_summary_rows.append({
                    'dataset': dataset_name,
                    'slug_used': slug,
                    'downloaded': True,
                    'local_ref_folder': str(target),
                    'source_cache_path': str(path),
                    'csv_files': len(csv_files),
                    'total_rows_csv': int(total_rows) if csv_files else None,
                    'min_cols': min(cols_list) if cols_list else None,
                    'max_cols': max(cols_list) if cols_list else None,
                    'sample_file': sample_file,
                    'sample_shape': sample_shape,
                    'error': ''
                })
                ok = True
                break
            except Exception as e:
                err_msg = str(e).splitlines()[0][:240]

        if not ok:
            shape_summary_rows.append({
                'dataset': dataset_name,
                'slug_used': None,
                'downloaded': False,
                'local_ref_folder': None,
                'source_cache_path': None,
                'csv_files': None,
                'total_rows_csv': None,
                'min_cols': None,
                'max_cols': None,
                'sample_file': None,
                'sample_shape': 'N/D',
                'error': err_msg or 'No se pudo descargar'
            })
else:
    for dataset_name in kaggle_targets:
        shape_summary_rows.append({
            'dataset': dataset_name,
            'slug_used': None,
            'downloaded': False,
            'local_ref_folder': None,
            'source_cache_path': None,
            'csv_files': None,
            'total_rows_csv': None,
            'min_cols': None,
            'max_cols': None,
            'sample_file': None,
            'sample_shape': 'N/D',
            'error': 'No hay credenciales de Kaggle detectadas'
        })

shape_summary_df = pd.DataFrame(shape_summary_rows)
print('\nResumen de descargas/shape para datasets Kaggle:')
display(shape_summary_df)

# 3) Exportar artefactos para actualizar el reporte
upfall_shapes_df.to_csv('analytics/upfall_local_shapes.csv', index=False)
shape_summary_df.to_csv('analytics/dataset_shapes_validation.csv', index=False)

print('\nArchivos generados:')
print('- analytics/upfall_local_shapes.csv')
print('- analytics/dataset_shapes_validation.csv')
print('- carpeta de referencia local:', download_root)

d:\UniversidadEAN\ProyectoGrado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kaggle auth detectado: True

UP-Fall local files detectados: 8


,dataset,file,rows,cols
0,UP-Fall (local),CameraResizedOF.csv,17378,805
1,UP-Fall (local),CompleteDataSet.csv,294679,47
2,UP-Fall (local),CompleteDataSetFeatures1&0.5.csv,32295,761
3,UP-Fall (local),CompleteDataSetFeatures2&1.csv,15593,761
4,UP-Fall (local),CompleteDataSetFeatures3&1.5.csv,10125,761
5,UP-Fall (local),Features_1&0.5_Vision.csv,32237,805
6,UP-Fall (local),Features_2&1_Vision.csv,15565,805
7,UP-Fall (local),Features_3&1.5_Vision.csv,10107,805



Resumen de descargas/shape para datasets Kaggle:


,dataset,slug_used,downloaded,local_ref_folder,source_cache_path,csv_files,total_rows_csv,min_cols,max_cols,sample_file,sample_shape,error
0,SisFall (mirror Kaggle),thevman/sisfall-dataset,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\thevman\sisfall-dataset\ve...,4275,6907663,10,10,SisFall_Task specified/Test/SA01/D01_SA01_R01.csv,"(5401, 10)",
1,KFall (mirror Kaggle),usmanabbasi2002/kfall-dataset,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\usmanabbasi2002\kfall-data...,5075,3995100,11,11,KFall Dataset/KFall Dataset/sensor_data/SA06/S...,"(3072, 11)",
2,FallAllD (mirror Kaggle),shusrith/fallalld,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\shusrith\fallalld\versions\5,5078,4101091,4,455,fallalld.csv,"(6605, 9)",
3,Real-Time Patient Fall,zara2099/real-time-patient-fall-detection-data,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\zara2099\real-time-patient...,1,1000,15,15,fall_detection_dataset.csv,"(1000, 15)",
4,Elderly Fall Detection IoT,ziya07/elderly-fall-detection-iot-dataset,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\ziya07\elderly-fall-detect...,2,25552,5,15,archive (14)/data_tuple3.csv,"(552, 5)",
5,Smartphone Human Fall Dataset,saadmansakib/smartphone-human-fall-dataset,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\saadmansakib\smartphone-hu...,2,1784,12,12,Test.csv,"(356, 12)",
6,Falls vs Normal Activities,enricogrimaldi/falls-vs-normal-activities,True,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,C:\khcache\datasets\enricogrimaldi\falls-vs-no...,1,96800,7,7,acc_gyr.csv,"(96800, 7)",



Archivos generados:
- analytics/upfall_local_shapes.csv
- analytics/dataset_shapes_validation.csv
- carpeta de referencia local: d:\UniversidadEAN\ProyectoGrado\datasets\dataset_downloads


## 7. Descarga de faltantes publicos y medicion de shapes

Esta seccion intenta descargar datasets no cubiertos previamente y genera un resumen de:
- descargas exitosas y fallidas,
- rutas locales,
- shapes cuando sea posible calcularlos.

In [2]:
from pathlib import Path
import json
import re
import csv
import requests
import pandas as pd

project_root = Path.cwd()
download_root = project_root / "datasets" / "dataset_downloads"
download_root.mkdir(exist_ok=True)


def csv_shape_fast(file_path: Path):
    def detect_delimiter(fp: Path) -> str:
        try:
            with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                sample = f.read(4096)
            dialect = csv.Sniffer().sniff(sample, delimiters=[',', ';', '\t', '|'])
            return dialect.delimiter
        except Exception:
            return ','

    delim = detect_delimiter(file_path)
    rows = 0
    cols = 0
    header = False
    with open(file_path, 'r', encoding='utf-8', errors='replace', newline='') as f:
        reader = csv.reader(f, delimiter=delim)
        for i, row in enumerate(reader):
            if i == 0:
                cols = len(row)
                header = any(not c.replace('.', '', 1).replace('-', '', 1).isdigit() for c in row if c != '')
            rows += 1
    if rows > 0 and header:
        rows -= 1
    return rows, cols


def summarize_local_tree(path: Path):
    csv_files = sorted(path.rglob('*.csv'))
    if not csv_files:
        return {
            'csv_files': 0,
            'total_rows_csv': None,
            'sample_file': None,
            'sample_shape': 'N/D'
        }
    total_rows = 0
    min_cols = None
    max_cols = None
    sample_file = None
    sample_shape = 'N/D'
    for i, fp in enumerate(csv_files):
        r, c = csv_shape_fast(fp)
        total_rows += int(r)
        min_cols = c if min_cols is None else min(min_cols, c)
        max_cols = c if max_cols is None else max(max_cols, c)
        if i == 0:
            sample_file = str(fp.relative_to(path)).replace('\\', '/')
            sample_shape = f'({r}, {c})'
    return {
        'csv_files': len(csv_files),
        'total_rows_csv': int(total_rows),
        'min_cols': min_cols,
        'max_cols': max_cols,
        'sample_file': sample_file,
        'sample_shape': sample_shape,
    }


def safe_download(url: str, out_path: Path, timeout=120):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


results = []

# A) UR Fall Detection: extraer links CSV directos de la pagina y bajar un subconjunto base
try:
    ur_dir = download_root / 'UR_Fall_Detection'
    ur_dir.mkdir(exist_ok=True)
    page = requests.get('http://fenix.ur.edu.pl/~mkepski/ds/uf.html', timeout=40)
    page.raise_for_status()
    html = page.text
    links = sorted(set(re.findall(r'href="([^"]+\\.(?:csv|zip))"', html, flags=re.IGNORECASE)))

    # Priorizar CSV de features y algunos CSV de sincronizacion/acelerometro
    preferred = [l for l in links if ('urfall-cam0' in l.lower() and l.lower().endswith('.csv'))]
    preferred += [l for l in links if ('acc.csv' in l.lower() or 'data.csv' in l.lower()) and l.lower().endswith('.csv')][:20]

    downloaded = 0
    for rel in preferred:
        full_url = rel if rel.startswith('http') else f'http://fenix.ur.edu.pl/~mkepski/ds/{rel.lstrip("./")}'
        out = ur_dir / Path(rel).name
        try:
            safe_download(full_url, out)
            downloaded += 1
        except Exception:
            continue

    summary = summarize_local_tree(ur_dir)
    results.append({
        'dataset': 'UR Fall Detection',
        'status': 'OK' if downloaded > 0 else 'PARTIAL/NO_CSV',
        'downloaded_files': downloaded,
        'local_folder': str(ur_dir),
        **summary,
        'note': 'Descargado subconjunto CSV (features + muestras).'
    })
except Exception as e:
    results.append({
        'dataset': 'UR Fall Detection',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'UR_Fall_Detection'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'Fallo: {str(e)[:220]}'
    })

# B) UniMiB SHAR: descargar raw CSV historico conocido
try:
    unimib_dir = download_root / 'UniMiB_SHAR'
    unimib_dir.mkdir(exist_ok=True)
    url_unimib = 'https://raw.githubusercontent.com/jindongwang/transferlearning/master/code/deep/Mobile/UniMiB_SHAR/data/full_data.csv'
    out_unimib = unimib_dir / 'full_data.csv'
    safe_download(url_unimib, out_unimib)
    r, c = csv_shape_fast(out_unimib)
    results.append({
        'dataset': 'UniMiB SHAR',
        'status': 'OK',
        'downloaded_files': 1,
        'local_folder': str(unimib_dir),
        'csv_files': 1,
        'total_rows_csv': int(r),
        'sample_file': 'full_data.csv',
        'sample_shape': f'({r}, {c})',
        'note': 'Mirror CSV historico descargado.'
    })
except Exception as e:
    results.append({
        'dataset': 'UniMiB SHAR',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'UniMiB_SHAR'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'Fallo: {str(e)[:220]}'
    })

# C) TST modificado (Zenodo): bajar .rar principal via API de Zenodo
try:
    tst_dir = download_root / 'TST_Fall_Modified_Zenodo'
    tst_dir.mkdir(exist_ok=True)
    z = requests.get('https://zenodo.org/api/records/3961894', timeout=40)
    z.raise_for_status()
    rec = z.json()
    files = rec.get('files', [])
    dl = None
    fname = None
    for f in files:
        links = f.get('links', {})
        if links.get('self'):
            dl = links.get('self')
            fname = f.get('key', 'zenodo_file.bin')
            break
    if dl:
        out = tst_dir / fname
        safe_download(dl, out, timeout=300)
        results.append({
            'dataset': 'TST Fall detection (modificado)',
            'status': 'OK',
            'downloaded_files': 1,
            'local_folder': str(tst_dir),
            'csv_files': 0,
            'total_rows_csv': None,
            'sample_file': fname,
            'sample_shape': 'N/D',
            'note': 'Descargado paquete .rar desde Zenodo; requiere extraccion para shapes internos.'
        })
    else:
        raise RuntimeError('No se encontro archivo descargable en API Zenodo')
except Exception as e:
    results.append({
        'dataset': 'TST Fall detection (modificado)',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'TST_Fall_Modified_Zenodo'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'Fallo: {str(e)[:220]}'
    })

# D) UMAFall y Graz via Figshare API
for ds_name, fig_id, folder_name in [
    ('UMAFall', 4214283, 'UMAFall_Figshare'),
    ('Graz (Mobile Phone Sensing Based Fall Detection)', 1444405, 'Graz_Figshare'),
]:
    try:
        ds_dir = download_root / folder_name
        ds_dir.mkdir(exist_ok=True)
        meta = requests.get(f'https://api.figshare.com/v2/articles/{fig_id}', timeout=40)
        meta.raise_for_status()
        article = meta.json()
        files = article.get('files', [])
        downloaded = 0
        for f in files:
            dl = f.get('download_url')
            name = f.get('name') or f.get('supplied_md5') or 'file.bin'
            if not dl:
                continue
            try:
                safe_download(dl, ds_dir / name, timeout=300)
                downloaded += 1
            except Exception:
                continue

        summary = summarize_local_tree(ds_dir)
        results.append({
            'dataset': ds_name,
            'status': 'OK' if downloaded > 0 else 'PARTIAL/EMPTY',
            'downloaded_files': downloaded,
            'local_folder': str(ds_dir),
            **summary,
            'note': 'Descarga por Figshare API; puede incluir zips/no-CSV.'
        })
    except Exception as e:
        results.append({
            'dataset': ds_name,
            'status': 'ERROR',
            'downloaded_files': 0,
            'local_folder': str(download_root / folder_name),
            'csv_files': None,
            'total_rows_csv': None,
            'sample_file': None,
            'sample_shape': 'N/D',
            'note': f'Fallo: {str(e)[:220]}'
        })

# E) tFall (Ultralytics): normalmente requiere login para export completo
try:
    tfall_dir = download_root / 'tFall_Ultralytics'
    tfall_dir.mkdir(exist_ok=True)
    info = tfall_dir / 'README_ACCESS.txt'
    info.write_text(
        'La pagina de Ultralytics del dataset tFall es accesible, pero la descarga completa suele requerir autenticacion/export desde plataforma.\nURL: https://platform.ultralytics.com/nicolai-nielsen/datasets/fall-detect\n',
        encoding='utf-8'
    )
    results.append({
        'dataset': 'tFall',
        'status': 'MANUAL_REQUIRED',
        'downloaded_files': 1,
        'local_folder': str(tfall_dir),
        'csv_files': 0,
        'total_rows_csv': None,
        'sample_file': 'README_ACCESS.txt',
        'sample_shape': 'N/D',
        'note': 'Descarga automatica no disponible sin login/export manual.'
    })
except Exception as e:
    results.append({
        'dataset': 'tFall',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'tFall_Ultralytics'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'Fallo: {str(e)[:220]}'
    })

missing_df = pd.DataFrame(results)
display(missing_df)
missing_df.to_csv('analytics/missing_public_datasets_downloads.csv', index=False)

print('Archivo generado: analytics/missing_public_datasets_downloads.csv')
print('Carpeta local de datasets:', download_root)

,dataset,status,downloaded_files,local_folder,csv_files,total_rows_csv,sample_file,sample_shape,note
0,UR Fall Detection,PARTIAL/NO_CSV,0,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0.0,None,NaN,N/D,Descargado subconjunto CSV (features + muestras).
1,UniMiB SHAR,ERROR,0,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,NaN,None,NaN,N/D,Fallo: 404 Client Error: Not Found for url: ht...
2,TST Fall detection (modificado),OK,1,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0.0,None,Fall-Detection-Dataset-Modification.rar,N/D,Descargado paquete .rar desde Zenodo; requiere...
3,UMAFall,OK,4,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0.0,None,NaN,N/D,Descarga por Figshare API; puede incluir zips/...
4,Graz (Mobile Phone Sensing Based Fall Detection),OK,1,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0.0,None,NaN,N/D,Descarga por Figshare API; puede incluir zips/...
5,tFall,MANUAL_REQUIRED,1,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0.0,None,README_ACCESS.txt,N/D,Descarga automatica no disponible sin login/ex...


Archivo generado: analytics/missing_public_datasets_downloads.csv
Carpeta local de datasets: d:\UniversidadEAN\ProyectoGrado\datasets\dataset_downloads


## 8. Resolucion de faltantes (UR, UniMiB, Graz, UMAFall)

Intento final para completar N/D:
- UR: descarga de CSV de features y series acc/data.
- UniMiB: descarga desde mirrors Kaggle detectados.
- Graz: lectura de SQLite para contar registros.
- UMAFall: inspeccion de ZIP para verificar tipo de contenido.

In [3]:
from pathlib import Path
import csv
import sqlite3
import zipfile
import requests
import pandas as pd
import kagglehub

project_root = Path.cwd()
download_root = project_root / 'datasets' / 'dataset_downloads'
download_root.mkdir(exist_ok=True)


def csv_shape_fast(file_path: Path):
    def detect_delimiter(fp: Path) -> str:
        try:
            with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                sample = f.read(4096)
            dialect = csv.Sniffer().sniff(sample, delimiters=[',', ';', '\t', '|'])
            return dialect.delimiter
        except Exception:
            return ','

    delim = detect_delimiter(file_path)
    rows = 0
    cols = 0
    header = False
    with open(file_path, 'r', encoding='utf-8', errors='replace', newline='') as f:
        reader = csv.reader(f, delimiter=delim)
        for i, row in enumerate(reader):
            if i == 0:
                cols = len(row)
                header = any(not c.replace('.', '', 1).replace('-', '', 1).isdigit() for c in row if c != '')
            rows += 1
    if rows > 0 and header:
        rows -= 1
    return rows, cols


def summarize_csv_tree(path: Path):
    csv_files = sorted(path.rglob('*.csv'))
    if not csv_files:
        return {'csv_files': 0, 'total_rows_csv': None, 'sample_file': None, 'sample_shape': 'N/D'}

    total_rows = 0
    min_cols = None
    max_cols = None
    sample_file = None
    sample_shape = 'N/D'

    for i, fp in enumerate(csv_files):
        r, c = csv_shape_fast(fp)
        total_rows += int(r)
        min_cols = c if min_cols is None else min(min_cols, c)
        max_cols = c if max_cols is None else max(max_cols, c)
        if i == 0:
            sample_file = str(fp.relative_to(path)).replace('\\', '/')
            sample_shape = f'({r}, {c})'

    return {
        'csv_files': len(csv_files),
        'total_rows_csv': int(total_rows),
        'min_cols': min_cols,
        'max_cols': max_cols,
        'sample_file': sample_file,
        'sample_shape': sample_shape,
    }


def safe_download(url: str, out_path: Path, timeout=120):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


rows = []

# 1) UR: descarga directa de CSV conocidos (features + series acc/data)
ur_dir = download_root / 'UR_Fall_Detection'
ur_dir.mkdir(exist_ok=True)
base = 'http://fenix.ur.edu.pl/~mkepski/ds/'
ur_urls = [
    base + 'urfall-cam0-falls.csv',
    base + 'urfall-cam0-adls.csv',
]
for i in range(1, 31):
    ur_urls.append(base + f'fall-{i:02d}-acc.csv')
    ur_urls.append(base + f'fall-{i:02d}-data.csv')
for i in range(1, 41):
    ur_urls.append(base + f'adl-{i:02d}-acc.csv')
    ur_urls.append(base + f'adl-{i:02d}-data.csv')

ok_ur = 0
for u in ur_urls:
    try:
        out = ur_dir / Path(u).name
        if out.exists() and out.stat().st_size > 0:
            ok_ur += 1
            continue
        safe_download(u, out, timeout=60)
        ok_ur += 1
    except Exception:
        continue

sum_ur = summarize_csv_tree(ur_dir)
rows.append({
    'dataset': 'UR Fall Detection',
    'status': 'OK' if ok_ur > 0 else 'ERROR',
    'downloaded_files': ok_ur,
    'local_folder': str(ur_dir),
    **sum_ur,
    'note': 'Descarga directa de CSV de features y secuencias (acc/data).'
})

# 2) UniMiB desde Kaggle mirrors detectados
unimib_dir = download_root / 'UniMiB_SHAR'
unimib_dir.mkdir(exist_ok=True)
unimib_slugs = [
    'wangboluo/unimib-shar-dataset',
    'alirezacman/unimib-shar',
    'panchadip0123892/unimib-shar',
    'sohomghosal/unimib-shar',
]

unimib_ok = False
unimib_err = ''
unimib_path = None
for slug in unimib_slugs:
    try:
        path = Path(kagglehub.dataset_download(slug))
        unimib_ok = True
        unimib_path = path
        (unimib_dir / 'SOURCE_PATH.txt').write_text(str(path), encoding='utf-8')
        rows.append({
            'dataset': 'UniMiB SHAR',
            'status': 'OK',
            'downloaded_files': 1,
            'local_folder': str(unimib_dir),
            **summarize_csv_tree(path),
            'note': f'Mirror Kaggle usado: {slug}'
        })
        break
    except Exception as e:
        unimib_err = str(e).splitlines()[0][:220]

if not unimib_ok:
    rows.append({
        'dataset': 'UniMiB SHAR',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(unimib_dir),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'No se pudo descargar via mirrors Kaggle: {unimib_err}'
    })

# 3) Graz (sqlite): contar tablas y filas
try:
    graz_db = download_root / 'Graz_Figshare' / 'dataset-falls-and-ADL.sqlite'
    if graz_db.exists():
        conn = sqlite3.connect(graz_db)
        cur = conn.cursor()
        tbls = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
        total = 0
        sample_table = None
        sample_shape = 'N/D'
        for t in tbls:
            n = cur.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
            total += int(n)
            if sample_table is None:
                sample_table = t
                cols = cur.execute(f'PRAGMA table_info("{t}")').fetchall()
                sample_shape = f'({n}, {len(cols)})'
        conn.close()
        rows.append({
            'dataset': 'Graz (Mobile Phone Sensing Based Fall Detection)',
            'status': 'OK',
            'downloaded_files': 1,
            'local_folder': str(download_root / 'Graz_Figshare'),
            'csv_files': 0,
            'total_rows_csv': total,
            'sample_file': sample_table,
            'sample_shape': sample_shape,
            'note': 'Dataset en SQLite; total de filas sumando tablas.'
        })
except Exception as e:
    rows.append({
        'dataset': 'Graz (Mobile Phone Sensing Based Fall Detection)',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'Graz_Figshare'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'No se pudo leer SQLite: {str(e)[:220]}'
    })

# 4) UMAFall: inspeccion de ZIP (no extraer completo)
try:
    uma_dir = download_root / 'UMAFall_Figshare'
    zips = sorted(uma_dir.glob('*.zip'))
    entries = 0
    sample = None
    for zp in zips:
        with zipfile.ZipFile(zp, 'r') as zf:
            names = zf.namelist()
            entries += len(names)
            if sample is None and names:
                sample = names[0]
    rows.append({
        'dataset': 'UMAFall',
        'status': 'OK',
        'downloaded_files': len(zips),
        'local_folder': str(uma_dir),
        'csv_files': 0,
        'total_rows_csv': None,
        'sample_file': sample,
        'sample_shape': 'N/D',
        'note': f'Paquetes ZIP descargados ({len(zips)}); entradas internas totales: {entries}.'
    })
except Exception as e:
    rows.append({
        'dataset': 'UMAFall',
        'status': 'ERROR',
        'downloaded_files': 0,
        'local_folder': str(download_root / 'UMAFall_Figshare'),
        'csv_files': None,
        'total_rows_csv': None,
        'sample_file': None,
        'sample_shape': 'N/D',
        'note': f'No se pudo inspeccionar ZIP: {str(e)[:220]}'
    })

resolve_df = pd.DataFrame(rows)
display(resolve_df)
resolve_df.to_csv('analytics/missing_public_datasets_downloads_round2.csv', index=False)
print('Archivo generado: analytics/missing_public_datasets_downloads_round2.csv')

100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 144M/144M [00:04<00:00, 36.2MB/s] 

Extracting files...


,dataset,status,downloaded_files,local_folder,csv_files,total_rows_csv,sample_file,sample_shape,note,min_cols,max_cols
0,UR Fall Detection,ERROR,0,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0,NaN,NaN,N/D,Descarga directa de CSV de features y secuenci...,NaN,NaN
1,UniMiB SHAR,OK,1,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,3,1777421.0,unimib_test.csv,"(177727, 7)",Mirror Kaggle usado: wangboluo/unimib-shar-dat...,7.0,7.0
2,Graz (Mobile Phone Sensing Based Fall Detection),OK,1,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0,637892.0,DEVICE_ALL_ACCELERATION,"(159300, 6)",Dataset en SQLite; total de filas sumando tablas.,NaN,NaN
3,UMAFall,OK,4,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0,NaN,UMAFall_Dataset/,N/D,Paquetes ZIP descargados (4); entradas interna...,NaN,NaN


Archivo generado: analytics/missing_public_datasets_downloads_round2.csv


In [4]:
# Intento alternativo UR Fall Detection via mirror GitHub (si el sitio oficial bloquea descarga automatica)
from pathlib import Path
import zipfile
import io
import requests
import pandas as pd

project_root = Path.cwd()
download_root = project_root / 'datasets' / 'dataset_downloads'
ur_mirror_dir = download_root / 'UR_Fall_GitHub_Mirror'
ur_mirror_dir.mkdir(exist_ok=True)

out = {
    'dataset': 'UR Fall Detection (GitHub mirror)',
    'status': 'ERROR',
    'local_folder': str(ur_mirror_dir),
    'csv_files': 0,
    'total_rows_csv': None,
    'sample_file': None,
    'sample_shape': 'N/D',
    'note': ''
}

try:
    zip_url = 'https://codeload.github.com/czhao30/UR-Fall-Detection-Dataset/zip/refs/heads/master'
    resp = requests.get(zip_url, timeout=120)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        zf.extractall(ur_mirror_dir)

    csv_files = sorted(ur_mirror_dir.rglob('*.csv'))
    total_rows = 0
    sample_file = None
    sample_shape = 'N/D'

    for i, fp in enumerate(csv_files):
        try:
            df_tmp = pd.read_csv(fp)
        except Exception:
            try:
                df_tmp = pd.read_csv(fp, encoding='latin1')
            except Exception:
                continue
        r, c = df_tmp.shape
        total_rows += int(r)
        if sample_file is None:
            sample_file = str(fp.relative_to(ur_mirror_dir)).replace('\\', '/')
            sample_shape = f'({r}, {c})'

    out.update({
        'status': 'OK' if len(csv_files) > 0 else 'PARTIAL',
        'csv_files': len(csv_files),
        'total_rows_csv': int(total_rows) if len(csv_files) > 0 else None,
        'sample_file': sample_file,
        'sample_shape': sample_shape,
        'note': 'Descargado desde mirror GitHub para mitigar bloqueo del host oficial.'
    })
except Exception as e:
    out['note'] = f'Fallo mirror GitHub: {str(e)[:220]}'

ur_mirror_df = pd.DataFrame([out])
display(ur_mirror_df)
ur_mirror_df.to_csv('analytics/ur_mirror_attempt.csv', index=False)
print('Archivo generado: analytics/ur_mirror_attempt.csv')

,dataset,status,local_folder,csv_files,total_rows_csv,sample_file,sample_shape,note
0,UR Fall Detection (GitHub mirror),ERROR,d:\UniversidadEAN\ProyectoGrado\dataset_downlo...,0,None,None,N/D,Fallo mirror GitHub: 404 Client Error: Not Fou...


Archivo generado: analytics/ur_mirror_attempt.csv


In [5]:
# Verificacion rapida de URLs espejo para UR
import requests
urls_test = [
    'https://codeload.github.com/czhao30/UR-Fall-Detection-Dataset/zip/master',
    'https://codeload.github.com/czhao30/UR-Fall-Detection-Dataset/zip/main',
]
for u in urls_test:
    try:
        r = requests.head(u, timeout=30, allow_redirects=True)
        print(u, '->', r.status_code)
    except Exception as e:
        print(u, '-> ERROR', str(e)[:120])

https://codeload.github.com/czhao30/UR-Fall-Detection-Dataset/zip/master -> 404
https://codeload.github.com/czhao30/UR-Fall-Detection-Dataset/zip/main -> 404


In [1]:
# Post-descompresion: metricas adicionales para cerrar N/D
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
download_root = project_root / 'datasets' / 'dataset_downloads'

rows = []

# UMAFall: contar filas/columnas reales de CSV ignorando metadatos con '%'
uma_root = download_root / 'UMAFall_Figshare'
uma_csvs = sorted(uma_root.rglob('*.csv'))
uma_total_rows = 0
uma_sample_shape = 'N/D'
uma_sample_file = None
uma_cols_set = set()

for i, fp in enumerate(uma_csvs):
    try:
        df = pd.read_csv(fp, sep=';', comment='%', header=None, engine='python')
        # Quitar columnas completamente vacias causadas por separador final
        df = df.dropna(axis=1, how='all')
        r, c = df.shape
        uma_total_rows += int(r)
        uma_cols_set.add(int(c))
        if i == 0:
            uma_sample_shape = f'({r}, {c})'
            uma_sample_file = str(fp.relative_to(uma_root)).replace('\\', '/')
    except Exception:
        continue

rows.append({
    'dataset': 'UMAFall',
    'csv_files': len(uma_csvs),
    'total_rows': int(uma_total_rows) if len(uma_csvs) else None,
    'min_cols': min(uma_cols_set) if uma_cols_set else None,
    'max_cols': max(uma_cols_set) if uma_cols_set else None,
    'sample_file': uma_sample_file,
    'sample_shape': uma_sample_shape,
})

# TST modificado: no CSV, contar videos .avi para cuantificar registros de archivos
tst_root = download_root / 'TST_Fall_Modified_Zenodo' / 'Fall-Detection-Dataset-Modification' / 'Fall-Detection-Dataset-Modification'
tst_avi = sorted(tst_root.rglob('*.avi')) if tst_root.exists() else []
rows.append({
    'dataset': 'TST Fall detection (modificado)',
    'csv_files': 0,
    'total_rows': None,
    'min_cols': None,
    'max_cols': None,
    'sample_file': str(tst_avi[0].relative_to(tst_root)).replace('\\', '/') if tst_avi else None,
    'sample_shape': 'N/D',
    'avi_files': len(tst_avi),
})

# UR: confirmar si sigue sin archivos descargables locales
ur_root = download_root / 'UR_Fall_Detection'
ur_csv = sorted(ur_root.rglob('*.csv')) if ur_root.exists() else []
rows.append({
    'dataset': 'UR Fall Detection',
    'csv_files': len(ur_csv),
    'total_rows': None,
    'min_cols': None,
    'max_cols': None,
    'sample_file': str(ur_csv[0].relative_to(ur_root)).replace('\\', '/') if ur_csv else None,
    'sample_shape': 'N/D',
})

post_df = pd.DataFrame(rows)
display(post_df)
post_df.to_csv('analytics/post_unzip_metrics.csv', index=False)
print('Archivo generado: analytics/post_unzip_metrics.csv')

,dataset,csv_files,total_rows,min_cols,max_cols,sample_file,sample_shape,avi_files
0,UMAFall,1492,9376668.0,7.0,7.0,UMAFall_Dataset/UMAFall_Dataset/UMAFall_Subjec...,"(6562, 7)",NaN
1,TST Fall detection (modificado),0,NaN,NaN,NaN,ADL-GRASP/Data1-ADL-grasp-1-Depth.avi,N/D,504.0
2,UR Fall Detection,0,NaN,NaN,NaN,NaN,N/D,NaN


Archivo generado: analytics/post_unzip_metrics.csv


In [2]:
# Resumen de tFall NDJSON (metadata + conteos)
import json
from collections import Counter
from pathlib import Path

nd_path = Path('datasets/dataset_downloads/tFall_Ultralytics/fall-detect.ndjson')
meta = None
image_count = 0
split_counts = Counter()
box_class_counts = Counter()
image_class_presence = Counter()
widths = []
heights = []

with nd_path.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        t = obj.get('type')
        if t == 'dataset' and meta is None:
            meta = obj
        elif t == 'image':
            image_count += 1
            split = obj.get('split', 'unknown')
            split_counts[split] += 1
            w = obj.get('width')
            h = obj.get('height')
            if isinstance(w, int):
                widths.append(w)
            if isinstance(h, int):
                heights.append(h)
            boxes = obj.get('annotations', {}).get('boxes', [])
            seen_in_image = set()
            for b in boxes:
                if isinstance(b, list) and len(b) >= 1:
                    cls = int(b[0])
                    box_class_counts[cls] += 1
                    seen_in_image.add(cls)
            for cls in seen_in_image:
                image_class_presence[cls] += 1

summary = {
    'dataset_name': meta.get('name') if meta else None,
    'task': meta.get('task') if meta else None,
    'bytes': meta.get('bytes') if meta else None,
    'class_names': meta.get('class_names') if meta else None,
    'version': meta.get('version') if meta else None,
    'created_at': meta.get('created_at') if meta else None,
    'updated_at': meta.get('updated_at') if meta else None,
    'image_records': image_count,
    'split_counts': dict(split_counts),
    'box_class_counts': dict(sorted(box_class_counts.items())),
    'image_class_presence': dict(sorted(image_class_presence.items())),
    'width_min': min(widths) if widths else None,
    'width_max': max(widths) if widths else None,
    'height_min': min(heights) if heights else None,
    'height_max': max(heights) if heights else None,
}

print(summary)


{'dataset_name': 'fall-detect', 'task': 'detect', 'bytes': 26766027, 'class_names': {'0': 'class0', '1': 'class1', '2': 'class2'}, 'version': 'latest', 'created_at': '2026-04-14T14:23:38.622Z', 'updated_at': '2026-04-28T19:04:56.268Z', 'image_records': 475, 'split_counts': {'train': 364, 'val': 111}, 'box_class_counts': {0: 284, 1: 134, 2: 134}, 'image_class_presence': {0: 284, 1: 104, 2: 117}, 'width_min': 170, 'width_max': 2720, 'height_min': 146, 'height_max': 4096}


In [3]:
# Prueba de acceso automatizable para UR Fall Detection (CSV clave)
import requests

candidates = [
    'http://fenix.ur.edu.pl/~mkepski/ds/',
    'https://fenix.ur.edu.pl/~mkepski/ds/',
    'http://fenix.ur.edu.pl/~mkepski/ds/data/',
    'https://fenix.ur.edu.pl/~mkepski/ds/data/',
]
files_to_test = [
    'fall-01-data.csv',
    'fall-01-acc.csv',
    'adl-01-data.csv',
    'adl-01-acc.csv',
    'urfall-cam0-falls.csv',
    'urfall-cam0-adls.csv',
]

results = []
for base in candidates:
    for fname in files_to_test:
        url = base + fname
        ok = False
        status = None
        size = None
        try:
            r = requests.get(url, timeout=20, stream=True)
            status = r.status_code
            ok = (status == 200)
            size = r.headers.get('Content-Length')
            r.close()
        except Exception as e:
            status = str(e)
        results.append({'base': base, 'file': fname, 'status': status, 'ok': ok, 'content_length': size})

import pandas as pd
probe_df = pd.DataFrame(results)
display(probe_df)
probe_df.to_csv('analytics/ur_download_probe.csv', index=False)
print('Archivo generado: analytics/ur_download_probe.csv')

,base,file,status,ok,content_length
0,http://fenix.ur.edu.pl/~mkepski/ds/,fall-01-data.csv,404,False,278
1,http://fenix.ur.edu.pl/~mkepski/ds/,fall-01-acc.csv,404,False,278
2,http://fenix.ur.edu.pl/~mkepski/ds/,adl-01-data.csv,404,False,278
3,http://fenix.ur.edu.pl/~mkepski/ds/,adl-01-acc.csv,404,False,278
4,http://fenix.ur.edu.pl/~mkepski/ds/,urfall-cam0-falls.csv,404,False,278
5,http://fenix.ur.edu.pl/~mkepski/ds/,urfall-cam0-adls.csv,404,False,278
6,https://fenix.ur.edu.pl/~mkepski/ds/,fall-01-data.csv,404,False,278
7,https://fenix.ur.edu.pl/~mkepski/ds/,fall-01-acc.csv,404,False,278
8,https://fenix.ur.edu.pl/~mkepski/ds/,adl-01-data.csv,404,False,278
9,https://fenix.ur.edu.pl/~mkepski/ds/,adl-01-acc.csv,404,False,278


Archivo generado: analytics/ur_download_probe.csv


In [4]:
# Descarga automatica liviana de UR: CSV tabulares (sin ZIP de imagenes)
from pathlib import Path
import requests
import pandas as pd

base = 'https://fenix.ur.edu.pl/~mkepski/ds/data/'
out_dir = Path('datasets/dataset_downloads/UR_Fall_Detection')
out_dir.mkdir(parents=True, exist_ok=True)

files = []
# Falls: 30 secuencias
for i in range(1, 31):
    s = f'{i:02d}'
    files.append(f'fall-{s}-data.csv')
    files.append(f'fall-{s}-acc.csv')
# ADL: 40 secuencias
for i in range(1, 41):
    s = f'{i:02d}'
    files.append(f'adl-{s}-data.csv')
    files.append(f'adl-{s}-acc.csv')
# Features extraidas globales
files.extend(['urfall-cam0-falls.csv', 'urfall-cam0-adls.csv'])

rows = []
for fname in files:
    url = base + fname
    target = out_dir / fname
    status = None
    downloaded = False
    size_bytes = None
    try:
        if target.exists() and target.stat().st_size > 0:
            status = 200
            downloaded = False
            size_bytes = target.stat().st_size
        else:
            r = requests.get(url, timeout=30)
            status = r.status_code
            if r.status_code == 200:
                target.write_bytes(r.content)
                downloaded = True
                size_bytes = len(r.content)
            else:
                size_bytes = None
    except Exception as e:
        status = str(e)
    rows.append({'file': fname, 'status': status, 'downloaded_now': downloaded, 'size_bytes': size_bytes})

ur_dl_df = pd.DataFrame(rows)
ok_df = ur_dl_df[ur_dl_df['status'] == 200]
print({'files_total': len(ur_dl_df), 'ok': int((ur_dl_df['status'] == 200).sum()), 'downloaded_now': int(ur_dl_df['downloaded_now'].sum())})
display(ok_df.head(10))
ur_dl_df.to_csv('analytics/ur_csv_download_log.csv', index=False)

# Shape resumido para UR descargado
shape_rows = []
total_rows = 0
for fp in sorted(out_dir.glob('*.csv')):
    try:
        df = pd.read_csv(fp)
        r, c = df.shape
        total_rows += int(r)
        shape_rows.append({'file': fp.name, 'rows': r, 'cols': c})
    except Exception as e:
        shape_rows.append({'file': fp.name, 'rows': None, 'cols': None, 'error': str(e)})

shape_df = pd.DataFrame(shape_rows)
shape_df.to_csv('analytics/ur_csv_shapes.csv', index=False)
print({'ur_csv_files_local': len(shape_df), 'ur_total_rows_csv': int(total_rows)})
print('Archivos generados: analytics/ur_csv_download_log.csv, analytics/ur_csv_shapes.csv')

{'files_total': 142, 'ok': 142, 'downloaded_now': 142}


,file,status,downloaded_now,size_bytes
0,fall-01-data.csv,200,True,2466
1,fall-01-acc.csv,200,True,9529
2,fall-02-data.csv,200,True,1631
3,fall-02-acc.csv,200,True,6379
4,fall-03-data.csv,200,True,3348
5,fall-03-acc.csv,200,True,15591
6,fall-04-data.csv,200,True,1412
7,fall-04-acc.csv,200,True,5766
8,fall-05-data.csv,200,True,2321
9,fall-05-acc.csv,200,True,9385


{'ur_csv_files_local': 142, 'ur_total_rows_csv': 45814}
Archivos generados: analytics/ur_csv_download_log.csv, analytics/ur_csv_shapes.csv
